# Step 1 — Synthetic Port Data Generation
**AI-Driven Berth Allocation System | MSc Artificial Intelligence | University of Hull**

---

## Overview

This notebook covers **Step 1** of the AI-BAS pipeline: generating a realistic synthetic dataset of port operations.

### Why Synthetic Data?
Real port AIS (Automatic Identification System) and berth scheduling data is commercially sensitive and not publicly available at the level of granularity required for this project. Following Bierwirth and Meisel (2015) and Dulebenets et al. (2019), we generate synthetic data **calibrated to published port statistics**, which is a well-established approach in Berth Allocation Problem (BAP) research.

### What this notebook produces
| File | Description |
|------|-------------|
| `data/synthetic/vessel_calls.csv` | 500 vessel arrivals with physical dimensions, ETA, delay, priority |
| `data/synthetic/weather.csv` | Hourly weather records (wind, wave, visibility, precipitation) |
| `data/synthetic/tides.csv` | Hourly tidal heights (sinusoidal M2 approximation) |
| `data/synthetic/berths.csv` | 5 berth configurations with physical specs and allowed cargo types |

> **Dissertation reference:** Chapter 3, Section 3.2

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Navigate to project root (one level up from notebooks/)
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    _root = os.path.dirname(_here)
else:
    _root = _here
os.chdir(_root)
sys.path.insert(0, _root)
print(f"Working directory: {os.getcwd()}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


from src.data.synthetic_generator import generate_and_save_all

print("Generating synthetic port data...")
vessels, weather, tides, berths = generate_and_save_all("data/synthetic")
print(f"Done. Generated {len(vessels):,} vessel records.")

## 1.1 Vessel Traffic Model

Vessel types are sampled from a categorical distribution reflecting typical multi-purpose terminal traffic. Physical dimensions (LOA, beam, draft) are drawn from type-specific ranges calibrated against the Lloyd's List vessel database.

**Five vessel types modelled:**

In [ ]:
print("=== VESSEL DATASET OVERVIEW ===")
otal vessel calls : 5,000
---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
Cell In[2], line 3
      1 print("=== VESSEL DATASET OVERVIEW ===")
      2 print(f"Total vessel calls : {len(vessels):,}")
----> 3 print(f"Date range         : {str(vessels['eta'].min())[:10]}  →  {str(vessels['eta'].max())[:10]}")
      4 print(f"Average delay      : {vessels['delay_hours'].mean():.2f} hours")
      5 print(f"Average service    : {vessels['service_hours'].mean():.2f} hours")
      6 print()

TypeError: 'Timestamp' object is not subscriptableprint(f"Total vessel calls : {len(vessels):,}")
print(f"Date range         : {str(vessels['eta'].min())[:10]}  →  {str(vessels['eta'].max())[:10]}")
print(f"Average delay      : {vessels['delay_hours'].mean():.2f} hours")
print(f"Average service    : {vessels['service_hours'].mean():.2f} hours")
print()
vessels[["vessel_id","vessel_type","length_m","beam_m",
         "draft_m","service_hours","delay_hours","priority"]].head(10)

### Vessel Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Type distribution
type_counts = vessels['vessel_type'].value_counts()
colors = ['#003366','#0070C0','#F0AB00','#70AD47','#ED7D31']
axes[0].bar(type_counts.index, type_counts.values, color=colors)
axes[0].set_title('Vessel Type Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Vessel Type')
axes[0].set_ylabel('Count')
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v+2, str(v), ha='center', fontweight='bold')

# Delay distribution
axes[1].hist(vessels['delay_hours'], bins=40, color='#003366', edgecolor='white', alpha=0.8)
axes[1].axvline(vessels['delay_hours'].mean(), color='#F0AB00', linewidth=2,
                label=f"Mean: {vessels['delay_hours'].mean():.2f}h")
axes[1].set_title('Arrival Delay Distribution', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Delay (hours)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Priority breakdown
pri_counts = vessels['priority'].value_counts()
axes[2].pie(pri_counts.values, labels=pri_counts.index,
            colors=['#C00000','#0070C0','#70AD47'],
            autopct='%1.1f%%', startangle=90)
axes[2].set_title('Priority Breakdown', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('notebooks/fig_step1_vessels.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 1.2 Berth Configuration

Five berths are defined with differentiated physical capabilities and cargo type restrictions. This heterogeneity is central to the Discrete BAP formulation — each vessel must be matched to a compatible berth.

In [ ]:
print("=== BERTH CONFIGURATION ===")
berths

## 1.3 Weather and Tidal Data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Wind speed distribution
axes[0].hist(weather['wind_speed_ms'], bins=35, color='#0070C0', edgecolor='white', alpha=0.8)
axes[0].set_title('Wind Speed Distribution (Weibull)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Wind Speed (m/s)')
axes[0].set_ylabel('Frequency')

# Tidal cycle — show one day
tides['timestamp'] = pd.to_datetime(tides['timestamp'])
one_day = tides[tides['timestamp'].dt.date == tides['timestamp'].dt.date.iloc[0]]
axes[1].plot(one_day['timestamp'], one_day['tide_height_m'],
             color='#003366', linewidth=2)
axes[1].fill_between(one_day['timestamp'], one_day['tide_height_m'],
                     alpha=0.2, color='#0070C0')
axes[1].axhline(6.5, color='#C00000', linestyle='--', linewidth=1.5,
                label='Low-tide threshold (6.5m)')
axes[1].set_title('Tidal Cycle — M2 Semidiurnal (one day)', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Tide Height (m)')
axes[1].legend()

plt.tight_layout()
plt.savefig('notebooks/fig_step1_environment.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Output | Value |
|--------|-------|
| Vessel records | 500 |
| Weather records | 2,160 hourly rows (90 days) |
| Tidal records | 2,160 hourly rows |
| Berths configured | 5 (B1–B5) |
| Target variable | `delay_hours` = ATA − ETA |

**Next step:** Run `02_feature_engineering.ipynb` to build the ML feature matrix.